# BrainTumNet Phase 2 – Full Pipeline Reference

## Model Architecture: SegUNetV2 Phase 2

**Training Command:**
```bash
python braintumnet/scripts/train.py --model segunetv2_phase2 --fold 4
```

This notebook contains the complete implementation of the BrainTumNet Phase 2 method for brain tumor segmentation, serving as the reference code for the research paper.

---

## Table of Contents
1. [Overview](#overview)
2. [Model Architecture Details](#architecture)
3. [Configuration](#configuration)
4. [Data Preprocessing](#preprocessing)
5. [Training Pipeline](#training)
6. [Evaluation & Inference](#evaluation)

---

## 1. Overview <a name="overview"></a>

### Model: SegUNetV2 Phase 2

**Architecture Type:** Enhanced U-Net with Multi-Scale Transformer and Attention Gates

**Key Features:**
- **Phase 1 Features (Inherited):**
  - InstanceNorm instead of BatchNorm (medical imaging standard)
  - LeakyReLU instead of ReLU (better gradients)
  - Residual connections in all encoder/decoder blocks
  - Strided convolution downsampling (learned, better than MaxPool)
  - CBAM attention on skip connections
  - Multi-scale fusion before final head
  - Boundary Refinement Module (improves IoU-Dice gap)
  - Deep supervision with auxiliary heads

- **Phase 2 New Features:**
  - **Multi-Scale Transformer Bottleneck** (patch sizes: 4, 8, 16)
  - **Attention Gates** for skip connections (nnU-Net style)

**Expected Performance:**
- Target Dice: 0.93-0.97
- Improvement over Phase 1: +2-4% Dice
- Improvement over Baseline: +6-11% Dice (total)

**Model Parameters:**
- Base channels: 64
- Transformer dimension: 512
- Transformer depth: 4
- Attention heads: 8
- Dropout: 0.2
- Total parameters: ~32-38M (estimated)

---

## 2. Model Architecture Details <a name="architecture"></a>

### 2.1 Overall Structure

The SegUNetV2 Phase 2 model consists of:
1. **Encoder** (4 levels with residual blocks)
2. **Multi-Scale Transformer Bottleneck** (Phase 2)
3. **Decoder** (4 levels with attention gates - Phase 2)
4. **Multi-Scale Fusion Module**
5. **Boundary Refinement Module** (Phase 1)
6. **Segmentation Head + Deep Supervision Heads**

### 2.2 Model Wrapper: BrainTumNetV2

The full model is wrapped in `BrainTumNetV2` which includes:
- Segmentation network: `SegUNetV2`
- Classification backbone: `TInceptionNet` (for HGG/LGG classification)
- ROI-guided classification with gradient control

**File:** `braintumnet/src/braintumnet/models/braintumnet_v2.py`

### 2.3 Encoder Architecture

**Component:** `EncoderBlock`

Each encoder block contains:
1. **ResidualConvBlock:**
   - Conv2d (3×3) → InstanceNorm → LeakyReLU → Dropout
   - Conv2d (3×3) → InstanceNorm
   - Residual connection (1×1 conv if channel mismatch)
   - Final LeakyReLU activation

2. **Strided Convolution Downsampling:**
   - Conv2d (3×3, stride=2) - learnable downsampling instead of MaxPool

**Encoder Levels:**
```
e1: 4 → 64 channels      (H×W → H×W, then H/2×W/2)
e2: 64 → 128 channels    (H/2×W/2 → H/2×W/2, then H/4×W/4)
e3: 128 → 256 channels   (H/4×W/4 → H/4×W/4, then H/8×W/8)
e4: 256 → 512 channels   (H/8×W/8 → H/8×W/8, then H/16×W/16)
```

**File:** `braintumnet/src/braintumnet/models/seg_unet_v2.py:82-99`

### 2.4 Multi-Scale Transformer Bottleneck (Phase 2)

**Component:** `MultiScaleTransformerBottleneck`

This is the core innovation of Phase 2, replacing the single-scale transformer.

**Architecture:**
1. **Multi-Scale Patch Embedding:**
   - Processes features at 3 different patch sizes: 4×4, 8×8, 16×16
   - Each scale captures different receptive fields
   - Projection: Conv2d with kernel_size=stride=patch_size
   - LayerNorm on flattened tokens

2. **Transformer Processing:**
   - Depth: 4 layers
   - Heads: 8
   - Each layer:
     - Multi-head Self-Attention (with residual)
     - MLP (expansion ratio 4.0, GELU activation, with residual)
   - Applied independently to each scale

3. **Cross-Scale Fusion:**
   - Upsample all scales to largest resolution (patch_size=4)
   - Concatenate along channel dimension
   - Linear projection: (dim × 3) → dim
   - LayerNorm → GELU → Dropout
   - Bilinear interpolation to restore spatial size

**Input/Output:**
- Input: (B, 512, H/16, W/16)
- Output: (B, 512, H/16, W/16)

**Expected Improvement:** +1.5-2.5% Dice through better global context

**File:** `braintumnet/src/braintumnet/models/multiscale_transformer.py`

### 2.5 Attention Gates (Phase 2)

**Component:** `AttentionGate`

Applied to skip connections in the decoder to suppress irrelevant features.

**Architecture:**
```
Inputs:
  g: gating signal from decoder (B, F_g, H, W)
  x: skip connection from encoder (B, F_l, H, W)

Processing:
  W_g: Conv2d(F_g → F_int) → InstanceNorm
  W_x: Conv2d(F_l → F_int) → InstanceNorm
  
  psi = LeakyReLU(W_g(g) + W_x(x))
  psi = Conv2d(F_int → 1) → Sigmoid
  
  output = x * psi  (element-wise multiplication)
```

**Parameters:**
- F_int = F_g / 2 (intermediate channels)

**Expected Improvement:** +1-2% Dice through better feature selection

**File:** `braintumnet/src/braintumnet/models/seg_unet_v2.py:102-151`

### 2.6 Decoder Architecture

**Component:** `DecoderBlock`

Each decoder block contains:
1. **Upsampling:**
   - ConvTranspose2d (kernel=2, stride=2)

2. **Attention Gate (Phase 2):**
   - Applied to skip connection
   - Uses decoder features as gating signal

3. **CBAM Attention (Phase 1):**
   - Channel attention + Spatial attention
   - Applied to skip connection after attention gate

4. **ResidualConvBlock:**
   - Operates on concatenated [upsampled, skip] features
   - Same structure as encoder blocks

**Decoder Levels:**
```
d4: 512 → 512 channels   (H/16×W/16 → H/8×W/8)
d3: 512 → 256 channels   (H/8×W/8 → H/4×W/4)  [aux_head3]
d2: 256 → 128 channels   (H/4×W/4 → H/2×W/2)  [aux_head2]
d1: 128 → 64 channels    (H/2×W/2 → H×W)      [aux_head1]
```

**File:** `braintumnet/src/braintumnet/models/seg_unet_v2.py:153-185`

### 2.7 Multi-Scale Fusion Module (Phase 1)

**Component:** `MultiScaleFusion`

Fuses decoder features from all levels for better multi-resolution understanding.

**Architecture:**
```
Inputs: [d1, d2, d3, d4] with channels [64, 128, 256, 512]

Processing:
  1. Project each to 64 channels (1×1 conv)
  2. Upsample all to d1 size (H×W) via bilinear interpolation
  3. Sum all features
  4. InstanceNorm → LeakyReLU
  
  5. Concatenate [d1, fused] → 128 channels
  6. ResidualConvBlock: 128 → 64 channels
```

**Output:** (B, 64, H, W) - enhanced features for segmentation head

**File:** `braintumnet/src/braintumnet/models/seg_unet_v2.py:250-292`

### 2.8 Boundary Refinement Module (Phase 1)

**Component:** `BoundaryRefinementModule`

Improves edge precision and reduces IoU-Dice gap.

**Architecture:**
```
1. Edge Detection:
   - Depthwise Conv2d (3×3, groups=in_channels)
   - Initialized with Sobel-like kernels
   - Learnable during training

2. Boundary Attention:
   Concat[features, edges] (C×2 channels)
   → Conv(C×2 → C) → InstanceNorm → LeakyReLU
   → Conv(C → C, 3×3) → InstanceNorm → LeakyReLU
   → Conv(C → C) → Sigmoid

3. Refinement:
   refined = features * (1 + attention)
```

**Expected Improvement:** +2-3% Dice, reduces IoU-Dice gap from 10% to ~5%

**File:** `braintumnet/src/braintumnet/models/seg_unet_v2.py:188-247`

### 2.9 Segmentation Head & Deep Supervision

**Main Head:**
- Conv2d (1×1): 64 → 4 channels (4-class segmentation)
- Output: (B, 4, H, W) logits

**Auxiliary Heads (Deep Supervision):**
- `aux_head3`: Conv2d (1×1): 256 → 4 channels (from d3)
- `aux_head2`: Conv2d (1×1): 128 → 4 channels (from d2)
- `aux_head1`: Conv2d (1×1): 64 → 4 channels (from d1)

**Deep Supervision Loss Weights:**
- Initial: 0.5 (aux_weight_initial)
- Final: 0.1 (aux_weight_final)
- Exponentially decayed during training

**File:** `braintumnet/src/braintumnet/models/seg_unet_v2.py:378-384`

### 2.10 Complete Forward Pass

```python
def forward(x):  # x: (B, 4, 256, 256)
    # Encoder
    s1, x1 = e1(x)      # s1: (B, 64, 256, 256),  x1: (B, 64, 128, 128)
    s2, x2 = e2(x1)     # s2: (B, 128, 128, 128), x2: (B, 128, 64, 64)
    s3, x3 = e3(x2)     # s3: (B, 256, 64, 64),   x3: (B, 256, 32, 32)
    s4, x4 = e4(x3)     # s4: (B, 512, 32, 32),   x4: (B, 512, 16, 16)
    
    # Bottleneck projection
    b = bottleneck_conv(x4)  # (B, 512, 16, 16)
    
    # Multi-Scale Transformer (Phase 2)
    b = multiscale_transformer(b)  # (B, 512, 16, 16)
    
    # Decoder with Attention Gates
    d4 = d4_block(b, s4)       # (B, 512, 32, 32)
    d3 = d3_block(d4, s3)      # (B, 256, 64, 64)
    d2 = d2_block(d3, s2)      # (B, 128, 128, 128)
    d1 = d1_block(d2, s1)      # (B, 64, 256, 256)
    
    # Multi-Scale Fusion
    fused = ms_fusion([d1, d2, d3, d4])  # (B, 64, 256, 256)
    combined = cat([d1, fused], dim=1)    # (B, 128, 256, 256)
    features = fusion_conv(combined)      # (B, 64, 256, 256)
    
    # Boundary Refinement
    features = boundary_refine(features)  # (B, 64, 256, 256)
    
    # Segmentation
    seg = head(features)  # (B, 4, 256, 256)
    
    # Deep supervision
    aux3 = aux_head3(d3)  # (B, 4, 64, 64)
    aux2 = aux_head2(d2)  # (B, 4, 128, 128)
    aux1 = aux_head1(d1)  # (B, 4, 256, 256)
    
    return seg, [aux3, aux2, aux1]
```

---

## 3. Configuration <a name="configuration"></a>

### 3.1 Model Configuration

**File:** `braintumnet/configs/models/segunetv2_phase2.yaml`

**Model Parameters:**
```yaml
model:
  model_type: "v2"
  in_channels: 4              # FLAIR, T1, T1CE, T2
  num_classes_seg: 4          # BG, NCR, ED, ET
  num_classes_cls: 2          # HGG/LGG
  
  # Architecture
  base: 64                    # Base channels (encoder: 64→128→256→512)
  dim: 512                    # Transformer dimension
  patch_size: 8               # Not used (multi-scale: 4,8,16)
  depth: 4                    # Transformer layers
  n_heads: 8                  # Attention heads
  dropout: 0.2                # Dropout rate
  
  # Features
  norm: "instance"            # InstanceNorm (medical imaging)
  roi_stop_grad: true         # Stop gradient in ROI path
  deep_supervision: true      # Auxiliary heads
  multi_scale_fusion: true    # Multi-scale fusion module
  
  # Phase 1 (inherited)
  boundary_refinement: true   # Boundary refinement module
  
  # Phase 2 (new)
  use_multiscale_transformer: true   # Multi-scale transformer (4,8,16)
  use_attention_gates: true          # Attention gates in decoder
```

### 3.2 Training Configuration

**Base Config:** `braintumnet/configs/base.yaml`

**Training Schedule:**
```yaml
train:
  epochs: 400
  batch_size: 12              # 3090 24GB (adjust for your GPU)
  val_batch_size: 16
  lr: 5.0e-5                  # AdamW learning rate
  weight_decay: 1.5e-4
  workers: 12
  
  # Optimizer
  optimizer: "adamw"
  grad_clip_norm: 1.0
  
  # Scheduler (Phase 2: SGDR)
  scheduler: "cosine_restarts"
  T_0: 50                     # Initial restart period
  T_mult: 2                   # Period multiplier
  min_lr: 1.0e-6
  warmup_steps: 3000
  
  # Mixed Precision
  amp: true
  amp_dtype: "float16"        # Use "bfloat16" for A100
  grad_accum_steps: 1
  
  # Phase 2: Gradient Centralization
  gradient_centralization: true
  
  # Training Control
  early_stop_patience: 50
  val_interval: 1
  compute_hd95: false         # Disable during training (expensive)
```

### 3.3 Loss Configuration (Phase 2)

**Loss Function:** Hybrid Dice + Focal + IoU + Boundary

```yaml
train:
  # Loss type
  loss_type: "dice_focal"
  seg_loss_weight: 1.0
  cls_loss_weight: 0.5        # Classification loss weight
  
  # Loss components
  dice_weight: 1.0
  focal_weight: 1.0
  iou_weight: 2.5
  boundary_weight: 1.0        # Phase 1 optimization
  
  # Focal Loss (4-class)
  focal_alpha: [0.0, 0.35, 0.35, 0.30]  # [BG, NCR, ED, ET]
  focal_gamma: 3.0
  
  # Class Weights (4-class)
  class_weights: [1.0, 2.5, 3.0, 4.0]   # [BG, NCR, ED, ET]
  ignore_background: true
  
  # Deep Supervision
  aux_weight_initial: 0.5     # Initial auxiliary loss weight
  aux_weight_final: 0.1       # Final auxiliary loss weight (decayed)
```

**Total Loss:**
```python
total_loss = (
    seg_loss_weight * (
        dice_weight * dice_loss +
        focal_weight * focal_loss +
        iou_weight * iou_loss +
        boundary_weight * boundary_loss
    ) +
    aux_weight * aux_loss +
    cls_loss_weight * cls_loss
)
```

### 3.4 Data Augmentation (Phase 2)

**Advanced medical imaging augmentations:**

```yaml
augment:
  # Geometric
  rotate_deg: 45
  hflip_p: 0.5
  vflip_p: 0.5
  scale_range: [0.9, 1.1]
  
  # Intensity
  brightness_range: [0.75, 1.25]
  contrast_range: [0.75, 1.25]
  gamma_range: [0.7, 1.4]
  gamma_p: 0.5
  
  # Noise
  gaussian_noise_p: 0.2
  gaussian_noise_std: 0.01
  gaussian_blur_p: 0.2
  gaussian_blur_sigma: [0.5, 1.5]
  
  # Advanced Medical Augmentation
  elastic_deform_p: 0.3       # Elastic deformation
  elastic_alpha: 30
  elastic_sigma: 4
  
  bias_field_p: 0.5           # MRI bias field simulation
  bias_field_scale: 0.3
  
  cutout_p: 0.2               # Cutout augmentation
  cutout_n_holes: 3
  cutout_size: 20
  
  local_shuffle_p: 0.15       # Local pixel shuffle
  local_shuffle_size: 3
```

---

## 4. Data Preprocessing <a name="preprocessing"></a>

### 4.1 Dataset: BraTS 2020

**Segmentation Classes (4-class):**
- 0: Background (BG)
- 1: Necrotic/Non-enhancing Tumor Core (NCR)
- 2: Peritumoral Edema (ED)
- 3: Enhancing Tumor (ET)

**Classification Classes:**
- 0: Low-Grade Glioma (LGG)
- 1: High-Grade Glioma (HGG)

### 4.2 Input Modalities

**4 MRI sequences:**
1. FLAIR (channel 0)
2. T1 (channel 1)
3. T1CE (channel 2) - contrast-enhanced
4. T2 (channel 3)

**Preprocessing:**
- Skull stripping
- N4 bias field correction
- Intensity normalization (z-score per modality)
- Resampling to 1mm³ isotropic
- Center cropping to 256×256
- Slice selection (tumor_slice_ratio=0.7)

### 4.3 Data Split

**5-Fold Cross-Validation:**
- Total cases: 369 (BraTS 2020)
- Per fold: ~74 validation, ~295 training
- Stratified by tumor grade (HGG/LGG)

**Current Fold:** 4

### 4.4 Data Loading

**Backend:** PNG (fast I/O)
- Alternative: LMDB for faster random access

**Paths:**
```yaml
data:
  raw_root: "braintumnet/data/raw/BraTS2020_TrainingData/MICCAI_BraTS2020_TrainingData"
  proc_root: "braintumnet/data/processed_multiclass_4class"
  lmdb_root: "braintumnet/data/lmdb_processed_multiclass_4class"
```

---

## 5. Training Pipeline <a name="training"></a>

### 5.1 Training Command

**Standard Training (Fold 4):**
```bash
python braintumnet/scripts/train.py --model segunetv2_phase2 --fold 4
```

**Training on A100 (optimized):**
```bash
python braintumnet/scripts/train.py --model segunetv2_phase2 --fold 4 --cfg a100
```

**Resume Training:**
```bash
python braintumnet/scripts/train.py --model segunetv2_phase2 --fold 4 --resume
```

### 5.2 Training All Folds

```bash
for fold in 0 1 2 3 4; do
    python braintumnet/scripts/train.py --model segunetv2_phase2 --fold $fold
done
```

### 5.3 Training Loop Details

**File:** `braintumnet/src/braintumnet/engine/trainer.py`

**Key Components:**
1. **DataLoader:**
   - num_workers: 12
   - pin_memory: True
   - persistent_workers: True
   - prefetch_factor: 4

2. **Optimizer:** AdamW
   - lr: 5e-5
   - weight_decay: 1.5e-4
   - gradient_centralization: True

3. **Scheduler:** CosineAnnealingWarmRestarts (SGDR)
   - T_0: 50 epochs
   - T_mult: 2
   - Warmup: 3000 steps

4. **Mixed Precision (AMP):**
   - dtype: float16 (or bfloat16 for A100)
   - GradScaler for stability

5. **Validation:**
   - Every epoch (val_interval=1)
   - Metrics: Dice, IoU (HD95 disabled during training)
   - Early stopping: patience=50

### 5.4 Checkpointing

**Saved Checkpoints:**
```
braintumnet/checkpoints/
├── best_fold4.pth          # Best validation Dice
├── last_fold4.pth          # Latest epoch (for resume)
└── v2_fold4_epoch*.pth     # Periodic saves (every 10 epochs)
```

**Checkpoint Contents:**
```python
{
    'epoch': int,
    'model_state_dict': OrderedDict,
    'optimizer_state_dict': dict,
    'scheduler_state_dict': dict,
    'best_dice': float,
    'best_iou': float,
    'config': dict,
}
```

### 5.5 Logging & Monitoring

**Log Files:**
```
braintumnet/logs/v2_fold4.log
```

**TensorBoard:**
```
braintumnet/runs/v2_fold4/
```

**Launch TensorBoard:**
```bash
tensorboard --logdir braintumnet/runs
```

**Logged Metrics:**
- Training: loss, dice, iou, lr
- Validation: dice, iou (per class and mean)
- Auxiliary losses (deep supervision)
- Learning rate schedule
- Sample predictions (images)

---

## 6. Evaluation & Inference <a name="evaluation"></a>

### 6.1 Evaluation

**Standard Evaluation:**
```bash
python braintumnet/scripts/evaluate.py \
  --checkpoint braintumnet/checkpoints/best_fold4.pth \
  --fold 4
```

**Metrics:**
- Dice Score (per class + mean)
- IoU (per class + mean)
- HD95 - Hausdorff Distance 95th percentile (boundary accuracy)
- Sensitivity, Specificity

**Output:**
```
Class-wise Metrics:
  NCR (1): Dice=0.XXX, IoU=0.XXX, HD95=X.XX mm
  ED  (2): Dice=0.XXX, IoU=0.XXX, HD95=X.XX mm
  ET  (3): Dice=0.XXX, IoU=0.XXX, HD95=X.XX mm
  
Mean Metrics:
  Dice: 0.XXX
  IoU:  0.XXX
  HD95: X.XX mm
```

### 6.2 Test-Time Augmentation (TTA)

**TTA Inference:**
```bash
python braintumnet/scripts/tta_inference.py \
  --checkpoint braintumnet/checkpoints/best_fold4.pth \
  --fold 4 \
  --tta_transforms hflip,vflip,rot90
```

**TTA Augmentations:**
- Horizontal flip
- Vertical flip  
- Rotation (90°, 180°, 270°)
- Multi-scale (0.9×, 1.0×, 1.1×)

**Expected Improvement:** +1-2% Dice with TTA

### 6.3 Ensemble Inference

**Ensemble All Folds:**
```bash
python braintumnet/scripts/ensemble_inference.py \
  --checkpoints \
    braintumnet/checkpoints/best_fold0.pth \
    braintumnet/checkpoints/best_fold1.pth \
    braintumnet/checkpoints/best_fold2.pth \
    braintumnet/checkpoints/best_fold3.pth \
    braintumnet/checkpoints/best_fold4.pth \
  --ensemble_method voting
```

**Ensemble Methods:**
- `voting`: Majority voting
- `averaging`: Average probabilities
- `weighted`: Weighted by validation Dice

**Expected Improvement:** +2-3% Dice with 5-fold ensemble

### 6.4 Visualization

**Visualize Predictions:**
```python
import matplotlib.pyplot as plt
from braintumnet.utils.visualization import plot_segmentation

# Load prediction
pred = ...
gt = ...
image = ...

# Plot
plot_segmentation(
    image=image,
    pred=pred,
    gt=gt,
    save_path="results/sample.png"
)
```

---

## Summary

### Model: SegUNetV2 Phase 2

**Command:**
```bash
python braintumnet/scripts/train.py --model segunetv2_phase2 --fold 4
```

**Key Features:**
1. **Multi-Scale Transformer Bottleneck** (patch sizes: 4, 8, 16)
2. **Attention Gates** in decoder skip connections
3. **Boundary Refinement Module** (Phase 1)
4. **Multi-Scale Fusion** (Phase 1)
5. **Deep Supervision** with auxiliary heads
6. **Advanced Augmentation** (elastic, bias field, etc.)
7. **SGDR Scheduler** with gradient centralization

**Architecture Files:**
- Main wrapper: `braintumnet/src/braintumnet/models/braintumnet_v2.py`
- Segmentation network: `braintumnet/src/braintumnet/models/seg_unet_v2.py`
- Multi-scale transformer: `braintumnet/src/braintumnet/models/multiscale_transformer.py`
- CBAM attention: `braintumnet/src/braintumnet/models/cbam.py`
- T-Inception (classification): `braintumnet/src/braintumnet/models/t_inception.py`

**Config Files:**
- Base: `braintumnet/configs/base.yaml`
- Model: `braintumnet/configs/models/segunetv2_phase2.yaml`

**Expected Performance:**
- Target Dice: 0.93-0.97
- Parameters: ~32-38M
- Training time: ~8-12 hours per fold (3090 24GB)